# Running Data Quality tests for DataFrame

In the following Notebook we will work with the `Tutorial Postgres.raw.public.taxi_yellow` table from the previous tutorial - [test workflow notebook](/lab/tree/notebooks/test_workflow.ipynb) - and perform some transformations on it before loading it into our staging database.

## Purpose
We want to showcase how we can hook OpenMetadata's data quality mechanisms directly in your ETLs before your data reaches its destination. For that, we're building an ETL that transforms the data we previously built and loads it in a table for which we have set up data quality tests in the [given instructions](/lab/tree/README.md).

## Description of the ETL
For context, please refer to [the test workflow notebook](/lab/tree/notebooks/test_workflow.ipynb)

In this case we will run some simple data cleaning and transformations on the dataframe. Then, we will use the `DataFrameValidator` interface to load the chunks of validated data into the destination and then finally report those results to OpenMetadata.

We will use the [`openmetadata-ingestion`](https://pypi.org/project/openmetadata-ingestion/) library to run the Data Quality tests we have defined in [OpenMetadata](http://localhost:8585/table/Tutorial%20Postgres.raw.public.taxi_yellow/profiler/data-quality).

## Dependencies
For our ETL we will be using SQLAlchemy to load the table, Pandas DataFrames to perform transformations, [`openmetadata-ingestion`](https://pypi.org/project/openmetadata-ingestion/) to run data quality tests and the OpenMetadata [Postgres Connector](https://docs.open-metadata.org/latest/connectors/database/postgres).

We can install all these dependencies specifying the right extras. A full list can be found in the project's [`setup.py`](https://github.com/open-metadata/OpenMetadata/blob/main/ingestion/setup.py), check it out if your installation differs from the example below.

## Requirements
If you haven't, please follow the [setup](/lab/tree/README.md#setup) steps in the README

For this example you will need:

- To have run the [`test_workflow`](/lab/tree/notebooks/test_workflow.ipynb) notebook
- An OpenMetadata instance running (achieved by following the setup instructions above)
- A bot JWT token. You can do so by using [Ingestion Bot's](http://localhost:8585/bots/ingestion-bot) token from your OpenMetadata instance
- [`openmetadata-ingestion`](https://pypi.org/project/openmetadata-ingestion/) version 1.11.0.0 or above (installed in this Notebook)

## Initial SDK setup
In this step we make sure our Python code is ready to work against OpenMetadata

Credentials are inherited from the environment (sourced by exec.sh).


In [ ]:
import os
from metadata.ingestion.ometa.ometa_api import OpenMetadata
from metadata.generated.schema.entity.services.connections.metadata.openMetadataConnection import OpenMetadataConnection
from metadata.generated.schema.api.lineage.addLineage import AddLineageRequest
from metadata.generated.schema.type.entityLineage import EntitiesEdge
from metadata.generated.schema.type.entityReference import EntityReference
from metadata.generated.schema.entity.data.table import Table

# 1. Credentials inherited from environment

# 2. Setup Connection (Str casting for Pydantic v2 strictness)
server_config = OpenMetadataConnection(
    hostPort=str(os.environ.get('API_COLLATE_BASE')),
    authProvider="openmetadata",
    securityConfig={"jwtToken": str(os.environ.get('TOKEN'))}
)
metadata = OpenMetadata(server_config)

## Implementation of the ETL

In [ ]:
# Define the transformation function to run on dataframes
def transform(df):
    # Keep only relevant columns
    cols_to_keep = [
        "vendorid", "tpep_pickup_datetime", "tpep_dropoff_datetime",
        "passenger_count", "trip_distance",
        "pulocationid", "dolocationid",
        "payment_type", "fare_amount", "tip_amount",
        "total_amount", "congestion_surcharge"
    ]
    df_stg = df[cols_to_keep]

    # Remove invalid or zero values
    df_stg = df_stg[
        (df_stg["fare_amount"] > 0) &
        (df_stg["total_amount"] > 0) &
        (df_stg["trip_distance"] > 0) &
        (df_stg["passenger_count"] > 0)
    ]

    # --- 2. Feature engineering ---
    df_stg["trip_duration_min"] = (
        (df_stg["tpep_dropoff_datetime"] - df_stg["tpep_pickup_datetime"]).dt.total_seconds() / 60
    )

    # Filter unrealistic durations and distances
    df_stg = df_stg[
        (df_stg["trip_duration_min"] >= 1) &
        (df_stg["trip_duration_min"] <= 180) &
        (df_stg["trip_distance"] <= 100)
    ]

    return df_stg

## Run Data Quality tests

In [ ]:
from metadata.sdk.data_quality.dataframes.dataframe_validator import DataFrameValidator

# 1. Initialize with your existing 'metadata' object
validator = DataFrameValidator(client=metadata)

# 2. Load the tests using the correct FQN
# Pattern: <Service>.<Database>.<Schema>.<Table>
# Based on your previous JSON, verify if it is postgres-taxi-stg or postgres-taxi-rides
table_fqn = "postgres-taxi-stg.stg.public.dw_taxi_trips"

print(f"📡 Fetching rules for {table_fqn}...")
validator.add_openmetadata_table_tests(table_fqn)

# 3. Run the "Shift-Left" validation on your transformed dataframe
# Assuming 'transformed_df' is the result of your transform(taxi_rides) function
# results = validator.validate(transformed_df)

# Alternatively, one could define the same tests as code with:
# from metadata.sdk.data_quality import ColumnValuesToBeBetween
# validator.add_tests(
#     ColumnValuesToBeBetween(
#         name="amount_is_greater_than_0",
#         min_value=0,
#     ),
#     ColumnValuesToBeBetween(
#         name="trip_duration_to_be_between_1_and_180_minutes",
#         min_value=1,
#         max_value=180,
#     ),
#     ColumnValuesToBeBetween(
#         name="trip_distance_to_be_at_most_100",
#         max_value=100,
#     ),
# )

### Run mechanisms

The `DataFrameValidator` is designed so that you can use it in a variety of use cases, including when memory is a concern and you're running your ETLs using Pandas. For such cases we have created a shortcut which will make your code smaller. But let's check the trivial use case first, when your whole data fits in memory.

#### Strategy: data fits in memory

The following ETL reads from the source, applies transformations and validates the dataframe

In [ ]:
# 1. Read from RAW
with source.connect() as conn:
    df = pd.read_sql("SELECT * FROM taxi_yellow", conn)

# 2. Transform 
df = transform(df)

# 3. Validate in-memory (using explicit client)
# Ensure your validator was initialized with: validator = DataFrameValidator(client=metadata)
results = validator.validate(df)

# 4. Load only if DQ passes
if results.success:
    print(f"✅ Validation Successful for {len(df)} rows. Loading to STG...")
    destination = create_engine(DESTINATION, future=True)

    with destination.connect() as conn:
        table = Table("dw_taxi_trips", MetaData(), autoload_with=conn)

        # Truncate and insert
        conn.execute(delete(table))
        conn.execute(insert(table), df.to_dict(orient="records"))
        conn.commit() 
    print("🚀 Load Complete.")
else:
    print(f"🛑 Load Aborted. {len(results.failures)} violations detected.")

# 5. Publish to the CORRECT Service
# We wrap the metadata client so the SDK finds the .ometa attribute it's looking for
from types import SimpleNamespace
wrapper = SimpleNamespace(ometa=metadata)

print(f"📊 Publishing results to {table_fqn}...")
results.publish("postgres-taxi-stg.stg.public.dw_taxi_trips", client=wrapper)
print("✅ Results successfully published to Collate.")

#### Strategy: loading data in chunks
Now, this use case has two variants. The first one is pretty similar to the one before and it requires that your code follows the validator's `FailureMode`, which defaults to a short circuit. The second requires that you only define three methods: one that returns chunks of probably transformed dataframes, another that loads chunks and a third that handles errors.

The validator's default and only (for now) failure mode short-circuits execution of any other test case and stops iterating on the chunks of data if a failure is encountered. We will want our code to behave as such, so after short circuiting we will rollback our changes in the destination database.

In [ ]:
# First: define a mechanism to load and transform chunks of data
## Credentials to the user are set up in `docker-compose.yml`

import pandas as pd
from sqlalchemy import create_engine

def load_and_transform():
    engine = create_engine(SOURCE)
    
    with engine.connect() as conn:
        chunks = pd.read_sql("SELECT * FROM taxi_yellow", conn, chunksize=1_000)

    for df in chunks:
        yield transform(df)

We will want the success and failure methods to have access to the same SQL connection so that everything stays in the same transaction. Thus we will create a little helper

In [ ]:
# A little helper to manage the connection
class SQLAlchemyValidationSession:
    def __init__(self, connection_string, table_name):
        self.engine = create_engine(connection_string, future=True)
        self.table = Table(table_name, MetaData(), autoload_with=self.engine)
        self._conn = None

    def with_conn(self, connection):
        self._conn = connection
        return self

    def load_df_to_destination(self, df, _result):
        """Loads data into destination."""
        self._conn.execute(insert(self.table).values(), df.to_dict(orient="records"))

    def rollback(self, _df, _result):
        """Clears data previously loaded"""
        self._conn.rollback()

    def __enter__(self):
        conn = self.engine.connect()
        return self.with_conn(conn)

    def __exit__(self ,type, value, traceback):
        self._conn.close()
        self._conn = None

**Example 1: loading data in chunks with the `DataFrameValidator.validate` method**

In [ ]:
# Example 1: loading data in chunks with the `DataFrameValidator.validate` method
from metadata.sdk.data_quality.dataframes.validation_results import ValidationResult

validation_session = SQLAlchemyValidationSession(
    connection_string=DESTINATION,
    table_name="dw_taxi_trips"
)

results = []
with validation_session as session:
    for transformed_df in load_and_transform():
        result = validator.validate(transformed_df)

        results.append(result)
        
        if result.success:
            session.load_df_to_destination(transformed_df, result)
        else:
            session.rollback(df, result)
            break

# Aggregate results for each test case for every chunk
results = ValidationResult.merge(*results)

# Publish to the CORRECT Service
# We wrap the metadata client so the SDK finds the .ometa attribute it's looking for
from types import SimpleNamespace
wrapper = SimpleNamespace(ometa=metadata)

print(f"📊 Publishing results to {table_fqn}...")
results.publish("postgres-taxi-stg.stg.public.dw_taxi_trips", client=wrapper)
print("✅ Results successfully published to Collate.")

**Example 2: loading data in chunks with the `DataFrameValidator.run` method**

This method is a shortcut to the loop above that returns results already merged

> ⚠ **NOTE:** there is however one caveat. Some of our tests require the whole dataframe to be in memory for them to work. For example, tests counting the total amount of rows would return false results because they'd be running on subsets of the data. Future versions of the SDK will solve this issue. For the time being, if your data does not fit in memory you should resort to the example in [test_workflow.ipynb](/lab/tree/notebooks/test_workflow.ipynb).

In [ ]:
with validation_session as session:
    results = validator.run(
        load_and_transform(),
        on_success=session.load_df_to_destination,
        on_failure=session.rollback,
    )

# Results are already merged and ready to publish in OpenMetadata
# We wrap the metadata client so the SDK finds the .ometa attribute it's looking for
from types import SimpleNamespace
wrapper = SimpleNamespace(ometa=metadata)

print(f"📊 Publishing results to {table_fqn}...")
results.publish("postgres-taxi-stg.stg.public.dw_taxi_trips", client=wrapper)
print("✅ Results successfully published to Collate.")

In [ ]:
# In both cases, results should be the same
for test_case, test_result in results.test_cases_and_results:

    print(f"\nTest: {test_case.name.root}")
    print(f"Status: {test_result.testCaseStatus}")
    print(f"Result: {test_result.result}")